# Diabetes Prediction — End-to-End Machine Learning Pipeline
This notebook builds a complete, reproducible Machine Learning workflow using `Diabetes_Prediction_Dataset.csv` (5,000 patient records, 13 features) and exports the best trained pipeline (`diabetes_model.joblib`) for deployment in the Streamlit web application.

## 1. Import Libraries & Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)

sns.set_theme(style="whitegrid", palette="muted")
df = pd.read_csv("Diabetes_Prediction_Dataset.csv")
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA) & Data Quality Audit

In [ ]:
# Inspect data types, missing values, and summary statistics
print("--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Summary Statistics ---")
display(df.describe())

print("\n--- Target Class Distribution ---")
print(df["Diabetes"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# Check categorical value distributions & potential anomalies
cat_cols = ["Gender", "Smoking_Status", "Physical_Activity", "Family_History"]
for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())

# Inspect extreme Blood_Pressure values (999 sentinel values)
print("\nRecords with Blood_Pressure > 250:", (df["Blood_Pressure"] > 250).sum())

In [ ]:
# Visualizations: Target balance & key clinical distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x="Diabetes", ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title("Target Variable Distribution (Diabetes)")

sns.boxplot(data=df, x="Diabetes", y="Glucose", ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Glucose Levels by Diabetes Status")

sns.boxplot(data=df, x="Diabetes", y="HbA1c", ax=axes[1, 0], palette="Set2")
axes[1, 0].set_title("HbA1c Levels by Diabetes Status")

sns.boxplot(data=df, x="Diabetes", y="BMI", ax=axes[1, 1], palette="Set2")
axes[1, 1].set_title("BMI Distribution by Diabetes Status")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical features
plt.figure(figsize=(10, 7))
num_df = df.drop(columns=["Patient_ID"]).copy()
num_df["Blood_Pressure"] = num_df["Blood_Pressure"].replace(999, np.nan)
num_df["Diabetes_Flag"] = num_df["Diabetes"].map({"Yes": 1, "No": 0})
corr = num_df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.show()

## 3. Data Cleaning & Preprocessing Pipeline
Key data cleaning findings addressed automatically inside the Scikit-Learn `Pipeline`:
1. Drop identifier column `Patient_ID`.
2. Replace sentinel/corrupted `Blood_Pressure == 999` values (47 records) with `NaN`.
3. Impute missing numerical values (`BMI`: 107 missing, `Glucose`: 70 missing, `Blood_Pressure`: 47 missing) using the **median** and apply `StandardScaler`.
4. Encode categorical variables (`Gender`, `Smoking_Status`, `Physical_Activity`, `Family_History`) using `OneHotEncoder(handle_unknown='ignore')`.

In [ ]:
# Clean dataset
df_clean = df.drop(columns=["Patient_ID"]).copy()
df_clean["Blood_Pressure"] = df_clean["Blood_Pressure"].replace(999, np.nan)

X = df_clean.drop(columns=["Diabetes"])
y = df_clean["Diabetes"].map({"Yes": 1, "No": 0})

numeric_features = ["Age", "BMI", "Glucose", "Blood_Pressure", "Cholesterol", "Insulin", "HbA1c"]
categorical_features = ["Gender", "Smoking_Status", "Physical_Activity", "Family_History"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

## 4. Model Training & Comparative Evaluation

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42)
}

results = []
trained_pipelines = {}

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "Test ROC-AUC": roc_auc_score(y_test, y_prob),
        "5-Fold CV AUC": cv_auc
    })

results_df = pd.DataFrame(results).sort_values(by="Test ROC-AUC", ascending=False)
display(results_df.style.format({col: "{:.4f}" for col in results_df.columns if col != "Model"}))

## 5. Best Model Diagnostics (Confusion Matrix, ROC Curve & Feature Importance)

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]
print(f"Selected Best Model: {best_model_name}")

y_pred_best = best_pipeline.predict(X_test)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best, target_names=["No Diabetes (0)", "Diabetes (1)"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"], ax=axes[0])
axes[0].set_title(f"Confusion Matrix — {best_model_name}")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

RocCurveDisplay.from_estimator(best_pipeline, X_test, y_test, ax=axes[1])
axes[1].set_title(f"ROC Curve — {best_model_name}")

plt.tight_layout()
plt.show()

In [ ]:
# Extract and plot feature importances
ohe_feature_names = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(ohe_feature_names)
importances = best_pipeline.named_steps["classifier"].feature_importances_

feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
feat_imp.tail(10).plot(kind="barh", color="#2b7bba")
plt.title(f"Top 10 Predictive Features ({best_model_name})")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

## 6. Export Serialized Model Artifact for Streamlit App

In [ ]:
model_artifact = {
    "pipeline": best_pipeline,
    "model_name": best_model_name,
    "features": numeric_features + categorical_features,
    "metrics": results_df.iloc[0].to_dict()
}

joblib.dump(model_artifact, "diabetes_model.joblib")
print("Saved 'diabetes_model.joblib' successfully!")